# Regulation resets: does the pace model forget fast enough?

A regulation change is a natural experiment (P7-01): the cars are new, so the history
before the change is a poor guide to the era after it. This notebook runs the P7-03 study
end to end: it walks the pace model forward across a reset, once per prior-handling
strategy (P7-02), and reads the recovery curve -- RPS by race number after the reset --
against the level the old era finished on.

It runs in CI from a clean environment (P7-04) against a **pinned fixture**, because CI has
no raw data and no network (ADR-0001). The data here is the synthetic world from
`tests/pace_world.py`; the study code is exactly the code the CLI runs.

In [ ]:
import os
import sys
from pathlib import Path

# Find the repository root (the directory with pyproject.toml) wherever the kernel starts,
# so `import tests.*` and the config paths resolve the same way the CLI runs them.
root = Path.cwd()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
root

In [ ]:
import matplotlib.pyplot as plt

from f1sim.adapters.config_files import load_natural_experiments
from f1sim.adapters.models.registry import ModelOptions
from f1sim.reports.recovery import STRATEGIES, STRATEGY_LABELS, render_recovery, run_recovery
from tests.pace_world import world_store

SEED = 20261012
experiment = next(e for e in load_natural_experiments() if e.experiment_id == "reg-2022")
experiment

## The pinned fixture

`world_store` builds a synthetic championship committed as one in-memory snapshot, so the
notebook is deterministic and needs no data lake. The seasons come from the manifest entry,
so the study covers the same shape of reset the CLI would run on real data.

In [ ]:
store = world_store(seasons=experiment.seasons, rounds=6)
snapshot = store.latest_snapshot()
snapshot

## Run the study

Every strategy sees the same races (one seed), differing only in how it treats the history
before the reset.

In [ ]:
study = run_recovery(
    store,
    snapshot=snapshot,
    experiment=experiment,
    strategies=STRATEGIES,
    seed=SEED,
    options=ModelOptions(),
    workers=2,
)
print(render_recovery(study))

## The curves

Lower is better. The dashed line is each strategy's pre-reset level: the race where a curve
drops below it is the race the model is back to where the old era left it.

In [ ]:
%matplotlib inline

fig, ax = plt.subplots(figsize=(7, 4))
for curve in study.curves:
    races = [p.race for p in curve.points]
    rps = [p.rps for p in curve.points]
    ax.plot(races, rps, marker="o", label=curve.label)
    if curve.pre_reset_rps is not None:
        ax.axhline(curve.pre_reset_rps, linestyle="--", linewidth=0.8, alpha=0.4)
ax.set_xlabel("Race number after the reset")
ax.set_ylabel("Ranked probability score")
ax.set_title(f"Recovery after {experiment.name} ({experiment.experiment_id})")
ax.legend()
fig.tight_layout()
plt.show()

## Checks

The notebook fails if the study stops producing a full curve for every strategy, so CI
catches a break instead of quietly publishing a stale result.

In [ ]:
assert [c.strategy for c in study.curves] == list(STRATEGIES)
assert all(c.pre_reset_rps is not None for c in study.curves)
assert all(len(c.points) == 6 for c in study.curves)
assert set(STRATEGY_LABELS) == set(STRATEGIES)
print(f"Fastest back to level: {study.fastest()}")

## Findings

The narrative write-up lives in `docs/research/regulation-reset-writeup.md`; the numbers the
CLI produces on the real snapshot are in `docs/research/regulation-reset.md`.